# 06 — 板块分析

> **先决条件**：`./trade data index sync-sector --start 2023-01-01`

| Section | 内容 |
|---------|------|
| 1 | 板块热力图（月/周涨跌）|
| 2 | 板块动量排名 |
| 3 | 板块轮动（相对沪深300）|
| 4 | 个股相对板块超额收益 |
| 5 | 板块传导可视化 |

In [ ]:
# ── 初始化 ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT / 'data'

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from trade_py.data.market.index.tushare import SW_SECTOR_INDICES
from trade_py.analysis.knowledge_graph import SW, SW_NAMES_ZH, SectorGraph

TODAY = date.today()
INDEX_DIR = DATA_ROOT / 'index'

print(f'Project: {PROJECT_ROOT}')
print(f'Today:   {TODAY}')

In [ ]:
# ── 加载所有板块指数 ──────────────────────────────────────────────────────────
sector_data = {}  # code → DataFrame

for code, (zh_name, sw_idx) in SW_SECTOR_INDICES.items():
    safe = 'sector_' + code.replace('.', '_') + '.parquet'
    p = INDEX_DIR / safe
    if p.exists():
        try:
            df = pd.read_parquet(p)
            df['_date'] = pd.to_datetime(df['date'])
            sector_data[code] = df.sort_values('_date').reset_index(drop=True)
        except Exception as e:
            print(f'加载失败 {code}: {e}')

print(f'已加载板块指数: {len(sector_data)}/31')
if len(sector_data) < 31:
    missing = [c for c in SW_SECTOR_INDICES if c not in sector_data]
    print(f'缺少: {missing[:5]}...' if len(missing) > 5 else f'缺少: {missing}')
    print('请运行: ./trade data index sync-sector --start 2023-01-01')

if sector_data:
    # 构建收益率 pivot 表（日频）
    close_dict = {}
    for code, df in sector_data.items():
        zh_name = SW_SECTOR_INDICES[code][0]
        close_dict[zh_name] = df.set_index('_date')['close']

    close_df = pd.DataFrame(close_dict).sort_index()
    ret_df = close_df.pct_change() * 100  # 日收益率%
    print(f'\n收益率矩阵: {ret_df.shape[0]} 天 × {ret_df.shape[1]} 板块')
    print(f'日期范围: {ret_df.index[0].date()} ~ {ret_df.index[-1].date()}')

## Section 1 — 板块热力图

In [ ]:
# 参数
PERIOD = 'W'   # 'W' = 周，'M' = 月
LOOKBACK_PERIODS = 52  # 显示最近 N 个周期

if not sector_data:
    print('⚠ 无板块数据')
else:
    # 按周/月汇总收益
    period_ret = close_df.resample(PERIOD).last().pct_change() * 100
    period_ret = period_ret.iloc[-LOOKBACK_PERIODS:].dropna(how='all')

    # A股配色：红=涨，绿=跌
    colorscale = [
        [0.0,  '#00c853'],   # 大跌 → 深绿
        [0.35, '#a5d6a7'],   # 小跌 → 浅绿
        [0.5,  '#f5f5f5'],   # 平盘
        [0.65, '#ef9a9a'],   # 小涨 → 浅红
        [1.0,  '#c62828'],   # 大涨 → 深红
    ]

    z = period_ret.T.values  # shape: (sectors, periods)
    x_labels = [str(d.date()) for d in period_ret.index]
    y_labels = list(period_ret.columns)

    vmax = float(np.nanpercentile(np.abs(z[np.isfinite(z)]), 95)) if np.isfinite(z).any() else 5

    fig = go.Figure(go.Heatmap(
        z=z,
        x=x_labels,
        y=y_labels,
        colorscale=colorscale,
        zmid=0,
        zmin=-vmax,
        zmax=vmax,
        colorbar=dict(title='涨跌幅%'),
        hoverongaps=False,
        hovertemplate='%{y}<br>%{x}<br>涨跌: %{z:.2f}%<extra></extra>',
    ))
    period_label = '周' if PERIOD == 'W' else '月'
    fig.update_layout(
        title=f'申万一级板块{period_label}度涨跌幅热力图（近{LOOKBACK_PERIODS}{period_label}）',
        xaxis_title=f'{period_label}份',
        yaxis_title='板块',
        height=700,
        xaxis=dict(tickangle=45),
    )
    fig.show()

## Section 2 — 板块动量排名

In [ ]:
if not sector_data:
    print('⚠ 无板块数据')
else:
    windows = [5, 20, 60]
    momentum = {}
    for w in windows:
        last_price = close_df.iloc[-1]
        past_price = close_df.iloc[-w - 1] if len(close_df) > w else close_df.iloc[0]
        momentum[f'{w}日涨幅%'] = ((last_price - past_price) / past_price * 100).round(2)

    mom_df = pd.DataFrame(momentum).sort_values('20日涨幅%', ascending=False)

    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=['近5日', '近20日', '近60日'])

    for i, (col, w) in enumerate(zip(['5日涨幅%', '20日涨幅%', '60日涨幅%'], windows), 1):
        df_w = mom_df.sort_values(col)
        colors = ['#ef5350' if v >= 0 else '#42a5f5' for v in df_w[col]]
        fig.add_trace(
            go.Bar(x=df_w[col], y=df_w.index, orientation='h',
                   name=f'{w}日', marker_color=colors, showlegend=False),
            row=1, col=i
        )

    fig.update_layout(title='板块动量排名', height=700)
    fig.show()

    # Top/Bottom 5
    for col in ['5日涨幅%', '20日涨幅%', '60日涨幅%']:
        print(f'\n{col} TOP5: {mom_df[col].nlargest(5).to_dict()}')
        print(f'{col} BOT5: {mom_df[col].nsmallest(5).to_dict()}')

## Section 3 — 板块轮动（相对沪深300）

In [ ]:
ROTATION_WINDOW = 60  # 相对强弱观察窗口
TRANSITION_LOOKBACK = 10  # 轮动信号：从弱→强的转换天数

hs300_path = INDEX_DIR / '000300_SH.parquet'

if not sector_data:
    print('⚠ 无板块数据')
elif not hs300_path.exists():
    print('⚠ 缺少沪深300数据，请先运行: ./trade data index sync')
else:
    hs300 = pd.read_parquet(hs300_path)
    hs300['_date'] = pd.to_datetime(hs300['date'])
    hs300 = hs300.set_index('_date')['close'].sort_index()

    # 对齐日期
    common_idx = close_df.index.intersection(hs300.index)
    close_aligned = close_df.loc[common_idx]
    hs300_aligned = hs300.loc[common_idx]

    # 相对强弱 = 板块/沪深300，近60日
    recent = close_aligned.iloc[-ROTATION_WINDOW:]
    hs300_recent = hs300_aligned.iloc[-ROTATION_WINDOW:]

    rel_strength = recent.divide(hs300_recent, axis=0)
    rel_strength_norm = rel_strength / rel_strength.iloc[0]  # 基于期初归一

    # 转强信号：最近10天相对强弱显著上升
    change = (rel_strength_norm.iloc[-1] - rel_strength_norm.iloc[-TRANSITION_LOOKBACK]) / rel_strength_norm.iloc[-TRANSITION_LOOKBACK] * 100
    rising = change.nlargest(8)

    fig = go.Figure()
    for col in close_df.columns:
        if col in rel_strength_norm.columns:
            fig.add_trace(go.Scatter(
                x=rel_strength_norm.index,
                y=rel_strength_norm[col],
                name=col,
                opacity=0.7,
                hovertemplate=f'{col}<br>%{{x}}<br>相对强弱: %{{y:.3f}}<extra></extra>',
            ))

    fig.update_layout(
        title=f'板块相对沪深300强弱（近{ROTATION_WINDOW}天，期初=1）',
        hovermode='x unified', height=600,
    )
    fig.show()

    print(f'\n📈 轮动信号（过去{TRANSITION_LOOKBACK}天相对强弱提升最多）:')
    for sector, val in rising.items():
        print(f'  {sector}: +{val:.1f}%')

## Section 4 — 个股相对板块超额收益

In [ ]:
# 自选股（留空则从 DB 取前20只）
WATCHLIST = []
EXCESS_WINDOW = 20  # 超额收益观察窗口（交易日）

import sqlite3

db_path = DATA_ROOT / '.metadata' / 'trade.db'
if not db_path.exists():
    print('⚠ trade.db 不存在')
else:
    with sqlite3.connect(str(db_path)) as conn:
        df_instr = pd.read_sql('SELECT symbol, name, industry FROM instruments WHERE industry > 0', conn)

    if not WATCHLIST:
        WATCHLIST = df_instr['symbol'].tolist()[:20]

    # symbol → sw_idx
    sym_to_ind = dict(zip(df_instr['symbol'], df_instr['industry']))
    # sw_idx → sector ts_code
    ind_to_code = {sw_idx: code for code, (zh_name, sw_idx) in SW_SECTOR_INDICES.items()}

    kline_dir = DATA_ROOT / 'kline'
    excess_rows = []

    for sym in WATCHLIST:
        kline_path = kline_dir / (sym.replace('.', '_') + '.parquet')
        if not kline_path.exists():
            continue
        try:
            kl = pd.read_parquet(kline_path)
            kl['_date'] = pd.to_datetime(kl['date'])
            kl = kl.sort_values('_date').tail(EXCESS_WINDOW + 1)
            if len(kl) < 2:
                continue
            stock_ret = (kl['close'].iloc[-1] - kl['close'].iloc[0]) / kl['close'].iloc[0] * 100

            sw_idx = sym_to_ind.get(sym, 0)
            sector_code = ind_to_code.get(sw_idx)
            sector_ret = 0.0
            sector_name = '未映射'
            if sector_code and sector_code in sector_data:
                sec_df = sector_data[sector_code].tail(EXCESS_WINDOW + 1)
                if len(sec_df) >= 2:
                    sector_ret = (sec_df['close'].iloc[-1] - sec_df['close'].iloc[0]) / sec_df['close'].iloc[0] * 100
                sector_name = SW_SECTOR_INDICES[sector_code][0]

            excess_ret = stock_ret - sector_ret
            excess_rows.append({
                'symbol': sym,
                'name': df_instr.set_index('symbol')['name'].get(sym, ''),
                'sector': sector_name,
                f'{EXCESS_WINDOW}日个股收益%': round(stock_ret, 2),
                f'{EXCESS_WINDOW}日板块收益%': round(sector_ret, 2),
                f'{EXCESS_WINDOW}日超额收益%': round(excess_ret, 2),
            })
        except Exception as e:
            pass

    if excess_rows:
        df_excess = pd.DataFrame(excess_rows).sort_values(f'{EXCESS_WINDOW}日超额收益%', ascending=False)
        display(df_excess)

        excess_col = f'{EXCESS_WINDOW}日超额收益%'
        fig = px.bar(
            df_excess, x='symbol', y=excess_col,
            color=excess_col,
            color_continuous_scale=['#42a5f5', '#f5f5f5', '#ef5350'],
            color_continuous_midpoint=0,
            title=f'个股近{EXCESS_WINDOW}日超额收益排名',
            hover_data=['name', 'sector'],
        )
        fig.show()
    else:
        print('无法计算超额收益（K线数据不足？）')

## Section 5 — 板块传导可视化

In [ ]:
# 选择触发板块（修改这里）
TRIGGER_SECTOR = SW.NonFerrousMetal  # 例：有色金属
TRIGGER_SCORE  = 1.0                 # 正 = 利好，负 = 利空
MAX_HOP = 2

graph = SectorGraph()
results = graph.propagate(TRIGGER_SECTOR, TRIGGER_SCORE, max_hop=MAX_HOP)

print(f'触发板块: {SW_NAMES_ZH[TRIGGER_SECTOR]}  (score={TRIGGER_SCORE:+.1f})')
print(f'传导结果 ({len(results)} 个受影响板块):\n')

rows = []
for r in results:
    # 结合实际近期涨跌（验证传导是否在近期发生）
    actual_ret = None
    sector_code = next(
        (c for c, (zh, sw_i) in SW_SECTOR_INDICES.items() if sw_i == int(r.sector)), None
    )
    if sector_code and sector_code in sector_data:
        sec_df = sector_data[sector_code].tail(6)
        if len(sec_df) >= 2:
            actual_ret = round((sec_df['close'].iloc[-1] - sec_df['close'].iloc[0]) / sec_df['close'].iloc[0] * 100, 2)

    rows.append({
        '板块': r.sector_name,
        '传导方向': '利好' if r.score > 0 else '利空',
        '传导强度': round(abs(r.score), 3),
        'Hop': r.hop,
        '关系类型': r.relation,
        '预期滞后(天)': r.typical_days,
        '近5日实际涨跌%': actual_ret if actual_ret is not None else '—',
        '路径': ' → '.join(r.path),
    })

df_prop = pd.DataFrame(rows)

def _color_direction(v):
    if v == '利好': return 'background-color: #ffcdd2'
    if v == '利空': return 'background-color: #bbdefb'
    return ''

display(df_prop.style.applymap(_color_direction, subset=['传导方向']))

In [ ]:
# 传导网络图（Plotly scatter with lines）
import plotly.graph_objects as go
import math

all_nodes = [SW_NAMES_ZH[TRIGGER_SECTOR]] + [r.sector_name for r in results]
node_idx = {n: i for i, n in enumerate(all_nodes)}

# 圆形布局
n = len(all_nodes)
angles = [2 * math.pi * i / n for i in range(n)]
xs = [math.cos(a) for a in angles]
ys = [math.sin(a) for a in angles]

edge_x, edge_y = [], []
for r in results:
    if len(r.path) >= 2:
        src_name = SW_NAMES_ZH.get(TRIGGER_SECTOR, r.path[-2].replace('SW_', ''))
        tgt_name = r.sector_name
        if src_name in node_idx and tgt_name in node_idx:
            si, ti = node_idx[src_name], node_idx[tgt_name]
            edge_x += [xs[si], xs[ti], None]
            edge_y += [ys[si], ys[ti], None]

node_colors = ['#ff7043'] + ['#ef5350' if r.score > 0 else '#42a5f5' for r in results]
node_sizes  = [20] + [max(8, int(abs(r.score) * 20)) for r in results]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
                          line=dict(color='gray', width=1), hoverinfo='none'))
fig.add_trace(go.Scatter(
    x=xs, y=ys, mode='markers+text',
    marker=dict(size=node_sizes, color=node_colors),
    text=all_nodes,
    textposition='top center',
    hoverinfo='text',
))
fig.update_layout(
    title=f'{SW_NAMES_ZH[TRIGGER_SECTOR]} 板块传导网络（红=利好 蓝=利空）',
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=600,
)
fig.show()